In [34]:
import polars as pl

# Just creating a fake DataFrame to experiment with
data = pl.DataFrame({
    'start': ["1/1/2020", "2/3/2020", "2/5/2020"],
    'end': ['5/20/2020', '3/4/2020', '8/19/2020']
    })

data

start,end
str,str
"""1/1/2020""","""5/20/2020"""
"""2/3/2020""","""3/4/2020"""
"""2/5/2020""","""8/19/2020"""


In [35]:
# First, let's convert the columns to dates. We could create a new column
# for this, but we'll just overwrite the old column since we're changing the type
# not the values
data = data.with_columns(
        pl.col('start').str.to_date('%m/%d/%Y'),
        pl.col('end').str.to_date('%m/%d/%Y')
    )
data

start,end
date,date
2020-01-01,2020-05-20
2020-02-03,2020-03-04
2020-02-05,2020-08-19


In [36]:
# Now, let's calculate the difference between the dates in days
data = data.with_columns(
    days = (pl.col('end') - pl.col('start')).dt.total_days()
)
data

start,end,days
date,date,i64
2020-01-01,2020-05-20,140
2020-02-03,2020-03-04,30
2020-02-05,2020-08-19,196


In [37]:
data.with_columns(
    new_start = pl.max_horizontal(pl.col('start'), pl.date(2020,2,1))
)

start,end,days,new_start
date,date,i64,date
2020-01-01,2020-05-20,140,2020-02-01
2020-02-03,2020-03-04,30,2020-02-03
2020-02-05,2020-08-19,196,2020-02-05


In [38]:
# Let's do a more complex calculation where we only start counting time after 2/1/2020
# So if their start date is before that, we use 2/1/2020 as our base, otherwise
# we use their actual start date
def calc_time_delta(start_col, end_col):
  return (end_col - pl.max_horizontal(start_col, pl.date(2020,2,1))).dt.total_days()

data = data.with_columns(
    days = calc_time_delta(pl.col('start'), pl.col('end'))
)
data

start,end,days
date,date,i64
2020-01-01,2020-05-20,109
2020-02-03,2020-03-04,30
2020-02-05,2020-08-19,196
